# CyberSentinel-EU Full Pipeline (v2, 1000-record dataset)
**Fatema Husain Hasan (202508958) MSc AI Thesis**

Runs on the expanded hand-labelled dataset (`labelling_master_1000.csv`).
Structured + Azure OpenAI embeddings -> hybrid features, with:
- structured vs embeddings vs hybrid comparison (H1)
- grouped CV by organisation (leakage-aware)
- leakage ablation (drop fine/articles/country)
- both tasks: cyber-vs-noncyber AND malicious-vs-nonmalicious

Works with ANY number of labelled rows - label some, run it, see scores improve.
Run top to bottom.

## 1. Install + import

In [1]:
!pip install openai --quiet

In [2]:
import pandas as pd, numpy as np, json, os
from openai import AzureOpenAI
print('ready')

ready


## 2. Azure credentials
Paste your endpoint + NEW key + deployment name. Clear the key before saving to GitHub.

In [3]:
AZURE_ENDPOINT = "https://cybersentinel-resource.openai.azure.com/"
AZURE_KEY = "PASTE-YOUR-KEY-HERE"
EMBED_DEPLOYMENT = "text-embedding-3-small"
API_VERSION = "2024-10-21"

client = AzureOpenAI(azure_endpoint=AZURE_ENDPOINT, api_key=AZURE_KEY, api_version=API_VERSION)
print("connection OK, vector length:", len(client.embeddings.create(model=EMBED_DEPLOYMENT, input="test").data[0].embedding))

connection OK, vector length: 1536


## 3. Load the labelled dataset
Upload your `labelling_master_1000.csv` to Colab (folder icon, left).
Only rows you've labelled are used; blanks are ignored automatically.

In [12]:
# 3. Load the labelled dataset + tracker (both from GitHub - no uploads needed)
REPO = "https://raw.githubusercontent.com/Fatimaxx24/202508958_IT9099_Thesis/main/"

full = pd.read_csv(REPO + "labelling_master_1000_labelled.csv")
tracker = pd.read_csv(REPO + "gdpr_enforcement_tracker_full.csv", low_memory=False)

# keep only labelled rows
lab = full[full['manual_label'].isin(['cyber_malicious','cyber_nonmalicious','not_cyber'])].copy()
lab['is_cyber'] = lab['manual_label'].str.startswith('cyber').astype(int)
print(f'labelled rows in use: {len(lab)}')
print(lab['manual_label'].value_counts().to_string())

# merge structured fields from tracker
data = lab.merge(tracker[['ETid','Controller/Processor','Quoted Articles']], on='ETid', how='left', suffixes=('','_t'))
# use the Summary already present in the master file
print('ready to embed:', len(data))

labelled rows in use: 891
manual_label
cyber_nonmalicious    401
not_cyber             296
cyber_malicious       194
ready to embed: 891


## 4. Embed the Summary text (cached)

In [13]:
CACHE = "embeddings_cache_1000.json"
cache = json.load(open(CACHE)) if os.path.exists(CACHE) else {}

def embed(text):
    k = str(text)[:8000]
    if k in cache: return cache[k]
    v = client.embeddings.create(model=EMBED_DEPLOYMENT, input=k).data[0].embedding
    cache[k] = v
    return v

vecs=[]
for i, s in enumerate(data['Summary'].astype(str)):
    vecs.append(embed(s))
    if (i+1)%50==0:
        print(f'{i+1}/{len(data)} embedded'); json.dump(cache, open(CACHE,'w'))
json.dump(cache, open(CACHE,'w'))
emb = np.array(vecs)
print('embeddings shape:', emb.shape)

50/891 embedded
100/891 embedded
150/891 embedded
200/891 embedded
250/891 embedded
300/891 embedded
350/891 embedded
400/891 embedded
450/891 embedded
500/891 embedded
550/891 embedded
600/891 embedded
650/891 embedded
700/891 embedded
750/891 embedded
800/891 embedded
850/891 embedded
embeddings shape: (891, 1536)


## 5. Feature engineering (structured / embeddings / hybrid)

In [14]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import PCA

data['year'] = pd.to_datetime(data['Date of Decision'], errors='coerce', format='mixed').dt.year
data['year'] = data['year'].fillna(data['year'].median())
data['log_fine'] = np.log1p(pd.to_numeric(data['Fine (EUR)'], errors='coerce')).fillna(-1)
data['n_articles'] = data['Quoted Articles'].astype(str).str.count('Art')

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
cat = ohe.fit_transform(data[['Country','Sector']].astype(str))
structured = np.hstack([cat, data[['year','log_fine','n_articles']].values])

# scale PCA components to dataset size
n_comp = min(100, emb.shape[0]-1, emb.shape[1])
pca = PCA(n_components=n_comp, random_state=42)
emb_pca = pca.fit_transform(emb)
print(f'PCA {pca.n_components_} comps, variance {pca.explained_variance_ratio_.sum():.1%}')

hybrid = np.hstack([structured, emb_pca])
print('structured:', structured.shape, '| emb:', emb_pca.shape, '| hybrid:', hybrid.shape)

PCA 100 comps, variance 77.2%
structured: (891, 44) | emb: (891, 100) | hybrid: (891, 144)


## 6. Task 1 Cyber vs Non-cyber (standard 5-fold)

In [16]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

y = data['is_cyber'].values
cv = StratifiedKFold(5, shuffle=True, random_state=42)
print("-CYBER vs NON-CYBER-")
for name, X in [('Structured', structured), ('Embeddings', emb_pca), ('Hybrid', hybrid)]:
    f1 = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=42), X, y, cv=cv, scoring='f1_macro')
    print(f'{name:12s} macro-F1 = {f1.mean():.3f} ± {f1.std():.3f}')

-CYBER vs NON-CYBER-
Structured   macro-F1 = 0.736 ± 0.038
Embeddings   macro-F1 = 0.795 ± 0.028
Hybrid       macro-F1 = 0.797 ± 0.028


## 7. Task 2 Malicious vs Non-malicious (cyber records only)

In [17]:
cyber = data[data['manual_label'].isin(['cyber_malicious','cyber_nonmalicious'])].copy()
idx = cyber.index
ym = (cyber['manual_label']=='cyber_malicious').astype(int).values
print(f'cyber records: {len(cyber)} | malicious: {ym.sum()} | non-malicious: {len(cyber)-ym.sum()}')

Xh = hybrid[[data.index.get_loc(i) for i in idx]]
Xs = structured[[data.index.get_loc(i) for i in idx]]
print("\n-MALICIOUS vs NON-MALICIOUS-")
for name, X in [('Structured', Xs), ('Hybrid', Xh)]:
    f1 = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=42), X, ym,
                         cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring='f1_macro')
    print(f'{name:12s} macro-F1 = {f1.mean():.3f} ± {f1.std():.3f}')

cyber records: 595 | malicious: 194 | non-malicious: 401

-MALICIOUS vs NON-MALICIOUS-
Structured   macro-F1 = 0.699 ± 0.031
Hybrid       macro-F1 = 0.713 ± 0.046


## 8. Grouped CV by organisation (leakage-aware) + leakage ablation

In [18]:
from sklearn.model_selection import cross_val_score, GroupKFold

groups = data['Controller/Processor'].astype(str).values
y = data['is_cyber'].values
gkf = GroupKFold(n_splits=5)

def grouped_f1(X):
    return cross_val_score(RandomForestClassifier(n_estimators=300, random_state=42),
                           X, y, cv=gkf, groups=groups, scoring='f1_macro')

print("-GROUPED CV (by organisation)-")
print(f'Hybrid grouped macro-F1 = {grouped_f1(hybrid).mean():.3f}')

# leakage ablation: remove fine + article-count (post-incident regulatory features)
# structured cols: [one-hot country/sector...][year][log_fine][n_articles]
struct_noreg = np.hstack([cat, data[['year']].values])  # drop fine + n_articles
hybrid_noreg = np.hstack([struct_noreg, emb_pca])
print("\n-LEAKAGE ABLATION-")
print(f'Hybrid (all features)        = {grouped_f1(hybrid).mean():.3f}')
print(f'Hybrid (no fine/articles)    = {grouped_f1(hybrid_noreg).mean():.3f}')
print("If the drop is small, the model is NOT relying on post-incident regulatory features (good).")

-GROUPED CV (by organisation)-
Hybrid grouped macro-F1 = 0.782

-LEAKAGE ABLATION-
Hybrid (all features)        = 0.782
Hybrid (no fine/articles)    = 0.783
If the drop is small, the model is NOT relying on post-incident regulatory features (good).


## 9. Notes
- Re-run any time after labelling more rows - scores update automatically.
- As the dataset grows, watch whether Hybrid overtakes Structured (H1).
- Next after labelling: SHAP explanations, temporal holdout, RAG stage.
- CLEAR AZURE_KEY before committing this notebook to GitHub.